<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Powerpredict" data-toc-modified-id="Powerpredict-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Powerpredict</a></span><ul class="toc-item"><li><span><a href="#Loading-the-dataset" data-toc-modified-id="Loading-the-dataset-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Loading the dataset</a></span></li></ul></li></ul></div>

# Powerpredict

Daily power consumption is related to the weather (rain, sunshine, temperature, etc). 
Prediction of the power consumption based on the weather is relevant for energy suppliers.
In this dataset you have to use the provided weather information to predict the power consumption.

In [1]:
import numpy
print(numpy.__version__)

1.22.4


## Loading the dataset

In [2]:
import pandas as pd
import os

In [3]:
DATASET_PATH = "."
if os.path.exists("/data/mlproject22"):
    DATASET_PATH = "/data/mlproject22"

In [4]:
powerpredict = pd.read_csv(os.path.join(DATASET_PATH,"powerpredict.csv.zip"))
X = powerpredict.drop(columns=["power_consumption"])
y = powerpredict[["power_consumption"]]

print(X.shape)
print(y.shape)

(39997, 65)
(39997, 1)


In [5]:
X

,Bedrock_t,Bedrock_t_low,Bedrock_t_high,Bedrock_bars,Bedrock_humidity,Bedrock_wind_speed,Bedrock_wind_deg,Bedrock_rain_short,Bedrock_rain_long,Bedrock_snow,...,Springfield_bars,Springfield_humidity,Springfield_wind_speed,Springfield_wind_deg,Springfield_rain_short,Springfield_rain_long,Springfield_snow,Springfield_clouds,Springfield_weather_main,Springfield_weather_description
0,270.475,270.475,270.475,1001,77,1,62,0.0,0.0,0.0,...,1035,100,7,58,0.0,0.0,0.0,0,clear,sky is clear
1,270.475,270.475,270.475,1001,77,1,62,0.0,0.0,0.0,...,1035,100,7,58,0.0,0.0,0.0,0,clear,sky is clear
2,269.686,269.686,269.686,1002,78,0,23,0.0,0.0,0.0,...,1036,100,7,48,0.0,0.0,0.0,0,clear,sky is clear
3,269.686,269.686,269.686,1002,78,0,23,0.0,0.0,0.0,...,1036,100,7,48,0.0,0.0,0.0,0,clear,sky is clear
4,269.686,269.686,269.686,1002,78,0,23,0.0,0.0,0.0,...,1036,100,7,48,0.0,0.0,0.0,0,clear,sky is clear
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39992,285.640,285.150,286.150,1028,62,2,140,0.0,0.0,0.0,...,1027,71,1,250,0.0,0.0,0.0,0,clear,sky is clear
39993,283.140,282.150,284.150,1029,71,1,242,0.0,0.0,0.0,...,1027,62,3,270,0.0,0.0,0.0,0,clear,sky is clear
39994,281.660,281.150,282.150,1029,81,3,300,0.0,0.0,0.0,...,1028,53,4,300,0.0,0.0,0.0,0,clear,sky is clear
39995,280.140,279.150,281.150,1029,81,2,310,0.0,0.0,0.0,...,1028,50,5,320,0.0,0.0,0.0,0,clear,sky is clear


Some columns are dropped here for simplicity, but they might provide useful information as well, so you might want to use them.

In [6]:
def drop_object_columns(df):
    drop_cols = [c for t, c in zip([t !="object" for t in df.dtypes], df.columns) if not t]
    return df.drop(columns=drop_cols)

DOC = drop_object_columns

def predict_show_metrics(name, reg, metric):
    print(f"{name}", metric(y, reg.predict(DOC(X))))


Here is an example dummy ML method showing the success of such a simple predictor:

In [7]:
import sklearn.metrics
import sklearn.dummy
reg = sklearn.dummy.DummyRegressor()
reg.fit(DOC(X), y)
metric = sklearn.metrics.mean_absolute_error

predict_show_metrics("Dummy", reg, metric)

Dummy 4085.9953308309014


You have to implement a simple method that performs the predictions with the given signature:

In [8]:
import joblib
import os

def leader_board_predict_fn(values):
    # 1. Define the path to your saved model file.
    # This assumes final_model.joblib is in the same directory as the notebook on JupyterHub.
    model_path = "final_model.joblib" 
    
    # 2. Load the trained pipeline (which includes both preprocessor and regressor)
    try:
        loaded_model = joblib.load(model_path)
    except FileNotFoundError:
        raise FileNotFoundError(f"Could not find the model file at {model_path}. Please ensure it is uploaded.")
    
    # 3. Generate and return predictions.
    # IMPORTANT: Do not use the DOC() function from the dummy example.
    # Your pipeline handles categorical data, so it needs the raw 'values' DataFrame.
    predictions = loaded_model.predict(values)
    
    return predictions

which will then be used to calculate the leaderboard score in a way similar to this:

In [9]:
def get_score():
    """
    Function to compute scores for train and test datasets.
    """
    
    import sklearn.metrics
    import pandas as pd
    import pathlib
    import os

    try:
        TEST_DATASET_PATH = "."
        if os.path.exists("/data/mlproject22/"):
            TEST_DATASET_PATH = "/data/mlproject22/"

        test_data = pd.read_csv(os.path.join(TEST_DATASET_PATH,"powerpredict.csv.zip"))
        X_test = test_data.drop(columns=["power_consumption"])
        y_test = test_data[["power_consumption"]]
        y_predicted = leader_board_predict_fn(X_test)
        dataset_score = sklearn.metrics.mean_absolute_error(y_test, y_predicted)
    except Exception:
        dataset_score = float("nan")
    print(f"Train Dataset Mean Absolute Error: {dataset_score}")

    import os
    import pwd
    import time
    import datetime
    import pandas as pd
    user_id = pwd.getpwuid( os.getuid() ).pw_name
    curtime = time.time()
    dt_now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

    try:  
        HIDDEN_DATASET_PATH = os.path.expanduser("/data/mlproject22-test-data/")
        test_data = pd.read_csv(os.path.join(HIDDEN_DATASET_PATH,"hidden_powerpredict.csv.zip"))
        X_test = test_data.drop(columns=["power_consumption"])
        y_test = test_data[["power_consumption"]]
        y_predicted = leader_board_predict_fn(X_test)
        hiddendataset_score = sklearn.metrics.mean_absolute_error(y_test, y_predicted)
        print(f"Test Dataset Mean Absolute Error: {hiddendataset_score}")
        score_dict = dict(
            score_hidden=hiddendataset_score,
            score_train=dataset_score,
            unixtime=curtime,
            user=user_id,
            dt=dt_now,
            comment="",
        )
    except Exception as e:
        err = str(e)
        score_dict = dict(
            score_hidden=float("nan"),
            score_train=dataset_score,
            unixtime=curtime,
            user=user_id,
            dt=dt_now,
            comment=err
        )

    #if list(pathlib.Path(os.getcwd()).parents)[0].name == 'source':
    #    print("we are in the source directory... replacing values.")
    #    print(pd.DataFrame([score_dict]))
    #    score_dict["score_hidden"] = -1
    #    score_dict["score_train"] = -1
    #    print("new values:")
    #    print(pd.DataFrame([score_dict]))

    pd.DataFrame([score_dict]).to_csv("powerpredict.csv", index=False)
    
get_score()

Train Dataset Mean Absolute Error: 1076.1166751053004
Test Dataset Mean Absolute Error: 2304.4244099597995
